### Random FASTQ Subsampling

In [ ]:
INPUT_READS = "manual_test/processed/EQA_04.noadapters.fastq" 
OUTPUT_READS = "manual_test/processed/EQA_04.subsampled.fastq"

SUBSAMPLE_AMOUNT = 10000
RANDOM_SEED = 42

In [ ]:
from pathlib import Path
import random

from Bio.SeqIO.QualityIO import FastqGeneralIterator

In [ ]:
# Count the total number of reads in the input FASTQ file.
with open(INPUT_READS) as input_handle:
    total_reads = sum(1 for _ in FastqGeneralIterator(input_handle))

# Avoid requesting more reads than are available.
sample_size = min(SUBSAMPLE_AMOUNT, total_reads)

# Create a reproducible random number generator.
rng = random.Random(RANDOM_SEED)

# Select unique read indices for subsampling.
selected_indices = set(rng.sample(range(total_reads), k=sample_size))

# Ensure the output directory exists.
Path(OUTPUT_READS).parent.mkdir(parents=True, exist_ok=True)

# Read the input FASTQ file and write only the selected reads.
with open(INPUT_READS) as input_handle, open(OUTPUT_READS, "w") as output_handle:
    for index, (name, sequence, quality) in enumerate(
        FastqGeneralIterator(input_handle)
    ):
        # Write the read if its index was selected.
        if index in selected_indices:
            output_handle.write(f"@{name}\n{sequence}\n+\n{quality}\n")

# Report the number of reads written.
print(f"Wrote {sample_size:,} of {total_reads:,} reads to {OUTPUT_READS}")